# Cloud Sharing: Protocols and Formats

This notebook documents the protocols, data formats, and architecture behind phasic's
cloud-based trace sharing system. The system allows pre-computed elimination traces to be
published, discovered, downloaded, and verified using content-addressed storage and a
GitHub-hosted registry.

## Architecture overview

The system has three layers:

| Layer | Component | Purpose |
|-------|-----------|--------|
| **Storage** | `TransportBackend` (ABC) | Content-addressed storage (IPFS, future: S3/GCS/Azure) |
| **Index** | `TraceRegistry` | GitHub-hosted JSON catalog with metadata, checksums, CIDs |
| **Client** | `get_trace()` / `TraceRegistry` | Download, cache, verify, and deserialize traces |

Data flows like this:

```
Publisher                         Consumer
--------                         --------
Graph -> record_elimination_trace()
       -> serialize to JSON
       -> gzip compress
       -> sha256 checksum                  get_trace("model_id")
       -> publish to IPFS (CID)                |
       -> submit PR to registry       registry.json (GitHub)
                                          |           |
                                        CID      checksum
                                          |           |
                                    IPFS/gateway  verify
                                          |           |
                                    trace.json.gz -> EliminationTrace
```

## 1. Transport Backend Protocol

The `TransportBackend` ABC defines the interface any storage backend must implement.
This decouples the registry from any specific storage technology.

In [ ]:
import inspect
from phasic.trace_repository import TransportBackend

# The abstract interface
for name, method in inspect.getmembers(TransportBackend, predicate=inspect.isfunction):
    if not name.startswith('_'):
        sig = inspect.signature(method)
        print(f"{name}{sig}")

### Required methods

| Method | Purpose |
|--------|---------|
| `name` (property) | Human-readable identifier, e.g. `"ipfs"`, `"s3"` |
| `get(cid, output_path=None)` | Retrieve content by identifier. Returns `bytes` or writes to file. |
| `add(path) -> str` | Publish local file/directory, returns content identifier. |

### Optional methods

| Method | Default | Purpose |
|--------|---------|--------|
| `supports_directories` (property) | `False` | Whether backend can retrieve whole directories |
| `get_directory(cid, output_dir)` | `NotImplementedError` | Retrieve entire directory tree |

### Implementing a custom backend

Any storage system can be plugged in by subclassing `TransportBackend`:

In [ ]:
from pathlib import Path
from phasic.trace_repository import TransportBackend

class LocalFileBackend(TransportBackend):
    """Example: serve traces from a local directory."""
    
    def __init__(self, root: Path):
        self.root = Path(root)
    
    @property
    def name(self) -> str:
        return "local"
    
    def get(self, cid, output_path=None):
        source = self.root / cid
        content = source.read_bytes()
        if output_path is not None:
            Path(output_path).write_bytes(content)
            return None
        return content
    
    def add(self, path):
        import shutil, hashlib
        data = Path(path).read_bytes()
        cid = hashlib.sha256(data).hexdigest()[:16]
        dest = self.root / cid
        shutil.copy(path, dest)
        return cid

# Use it with TraceRegistry:
# registry = TraceRegistry(backend=LocalFileBackend("/path/to/traces"))

### The default IPFS backend

`IPFSBackend` implements `TransportBackend` with a progressive enhancement strategy:

1. **Try existing daemon** &mdash; fastest, uses `ipfshttpclient` library
2. **Auto-start daemon** &mdash; if IPFS is installed but not running
3. **HTTP gateways** &mdash; falls back to public gateways (ipfs.io, cloudflare, etc.)

Content identifiers (CIDs) come in two formats:

| Format | Pattern | Example |
|--------|---------|--------|
| CIDv0 | `Qm` + 44 base58 chars | `QmYwAPJzv5CZsnN625s3Xf2nemtYgPpHdWEz79ojWnPbdG` |
| CIDv1 | `bafy` + base32 | `bafybeigdyrzt5sfp7udm7hu76uh7y26nf3efuylqabf3oclgtqy55fbzdi` |

CIDs can include subpaths: `QmYwAP.../trace.json.gz` to reference files within a directory.

In [ ]:
from phasic.trace_repository import _validate_cid

# Valid CIDs
_validate_cid("QmYwAPJzv5CZsnN625s3Xf2nemtYgPpHdWEz79ojWnPbdG")  # CIDv0
_validate_cid("bafybeigdyrzt5sfp7udm7hu76uh7y26nf3efuylqabf3oclgtqy55fbzdi")  # CIDv1
_validate_cid("QmYwAPJzv5CZsnN625s3Xf2nemtYgPpHdWEz79ojWnPbdG/trace.json.gz")  # subpath
print("All valid CIDs accepted.")

# Invalid CIDs are rejected
import traceback
for bad_cid in ["", "not-a-cid", "QmYwAPJzv5CZsnN625s3Xf2nemtYgPpHdWEz79ojWnPbdG/../../../etc/passwd"]:
    try:
        _validate_cid(bad_cid)
    except ValueError as e:
        print(f"Rejected {bad_cid!r}: {e}")

## 2. Registry Format (`registry.json`)

The registry is a JSON file hosted on GitHub that maps human-readable trace IDs
to IPFS CIDs, checksums, and metadata. It is the index that makes traces discoverable.

**Location**: `https://raw.githubusercontent.com/{repo}/master/registry.json`

### Schema

```json
{
  "version": "1.0.0",                    // REQUIRED: registry format version
  "updated": "2025-10-21T12:00:00Z",     // last update timestamp
  "traces": {                             // REQUIRED: dict of trace_id -> entry
    "coalescent_n5_theta1": {
      "cid": "QmStRuZZ...",              // IPFS directory CID
      "description": "Kingman coalescent, n=5, 1 parameter",
      "checksum": "sha256:abcdef01...",   // SHA-256 of trace.json.gz
      "graph_hash": "a1b2c3d4...",        // graph structure hash (for auto-discovery)
      "metadata": {
        "model_type": "coalescent",
        "domain": "population-genetics",
        "param_length": 1,
        "vertices": 5,
        "tags": ["coalescent", "kingman"]
      },
      "files": {
        "trace.json.gz":  {"cid": "QmStRuZZ.../trace.json.gz"},
        "metadata.json":  {"cid": "QmStRuZZ.../metadata.json"}
      },
      "license": "MIT"
    }
  },
  "collections": {                        // optional groupings
    "coalescent_basic": {
      "description": "Basic coalescent models",
      "traces": ["coalescent_n3_theta1", "coalescent_n5_theta1"]
    }
  }
}
```

### Registry validation

The registry is validated on load. Required top-level keys are `version` and `traces`,
and `traces` must be a dict:

In [ ]:
from phasic.trace_repository import _validate_registry
from phasic.exceptions import PTDBackendError

# Valid
_validate_registry({"version": "1.0", "traces": {}})
print("Minimal valid registry accepted.")

# Invalid examples
for bad in [
    "not a dict",
    {"version": "1.0"},           # missing 'traces'
    {"version": "1.0", "traces": "not-a-dict"},
]:
    try:
        _validate_registry(bad)
    except PTDBackendError as e:
        print(f"Rejected: {e}")

## 3. Trace Serialization Format

Traces are stored as gzipped JSON files (`trace.json.gz`). The JSON encodes an
`EliminationTrace` object: the complete record of graph elimination operations
that can be replayed with different parameter values.

### Required keys

These keys must be present in every serialized trace:

In [ ]:
from phasic.trace_repository import _REQUIRED_TRACE_KEYS, TRACE_FORMAT_VERSION

print(f"Trace format version: {TRACE_FORMAT_VERSION}")
print(f"\nRequired keys ({len(_REQUIRED_TRACE_KEYS)}):")
for key in sorted(_REQUIRED_TRACE_KEYS):
    print(f"  - {key}")

### Complete JSON schema

```json
{
  // --- Required ---
  "operations": [
    {
      "op_type": "CONST",                // OpType enum name: CONST|PARAM|DOT|ADD|MUL|DIV|INV|SUM
      "operands": [],                     // indices into the operations list
      "const_value": 1.0,                 // for CONST ops
      "param_idx": null,                  // for PARAM ops: which parameter theta[i]
      "coefficients": null                // for DOT ops: [c1, c2, ..., cn]
    },
    {"op_type": "PARAM", "operands": [], "param_idx": 0},
    {"op_type": "DOT",   "operands": [], "coefficients": [3.0, 1.5]},
    {"op_type": "ADD",   "operands": [1, 2]},
    {"op_type": "MUL",   "operands": [0, 3]}
  ],
  "vertex_rates": [0, 3, 4],             // operation index for each vertex's rate
  "edge_probs":   [[1, 2], [3], []],     // per-vertex list of op indices for edge weights
  "vertex_targets": [[1, 2], [2], []],   // per-vertex list of target vertex indices
  "states": [[0], [5], [1]],             // vertex state vectors (n_vertices x state_length)
  "starting_vertex_idx": 0,              // index of starting vertex
  "n_vertices": 3,                       // total number of vertices
  "param_length": 2,                     // number of free parameters
  "state_length": 1,                     // dimension of state vectors

  // --- Optional ---
  "vertex_indices": [0, 3, 7],           // original vertex indices from source graph
  "reward_length": 0,                    // number of reward parameters
  "is_discrete": false,                  // continuous (PH) vs discrete (DPH)
  "metadata": {                          // arbitrary metadata
    "phase": 2,
    "parameterized": true,
    "total_operations": 5
  }
}
```

### Operation types

The `op_type` field references the `OpType` enum. Each operation is an instruction
in a linear trace that can be replayed to evaluate the eliminated graph:

| OpType | Meaning | Key fields |
|--------|---------|------------|
| `CONST` | Literal constant | `const_value` |
| `PARAM` | Parameter reference &theta;[i] | `param_idx` |
| `DOT`   | Linear combination c&#8321;&theta;&#8321; + c&#8322;&theta;&#8322; + ... | `coefficients` |
| `ADD`   | a + b | `operands` = [idx_a, idx_b] |
| `MUL`   | a &times; b | `operands` = [idx_a, idx_b] |
| `DIV`   | a / b | `operands` = [idx_a, idx_b] |
| `INV`   | 1 / a | `operands` = [idx_a] |
| `SUM`   | &Sigma;(a, b, c, ...) | `operands` = [idx_a, idx_b, ...] |

The `operands` list contains integer indices pointing to earlier operations in the
same `operations` array, forming a DAG evaluated in order.

## 4. Working example: build, serialize, deserialize

Let's walk through the full lifecycle: build a graph, record the elimination trace,
serialize it to the trace format, and deserialize it back.

In [ ]:
import numpy as np
from phasic import Graph, with_ipv
from phasic.trace_elimination import record_elimination_trace

# Build a simple coalescent graph for n=3 samples.
# The @with_ipv decorator specifies the initial probability vector:
# state = [3] means 3 lineages in one population.
nr_samples = 3

@with_ipv([nr_samples])
def coalescent_callback(state):
    n = state[0]
    if n <= 1:
        return []
    rate = n * (n - 1) / 2
    return [[np.array([n - 1]), [rate]]]

graph = Graph(coalescent_callback, theta_dim=1)

print(f"Graph: {graph.vertices_length()} vertices")

In [ ]:
# Record the elimination trace
trace = record_elimination_trace(graph, theta_dim=1)
print(trace)
print(trace.summary())

In [ ]:
import json

def serialize_trace(trace):
    """Convert EliminationTrace to a JSON-serializable dict."""
    def to_native(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, list):
            return [to_native(x) for x in obj]
        elif isinstance(obj, dict):
            return {k: to_native(v) for k, v in obj.items()}
        return obj

    return {
        'operations': [
            {
                'op_type': op.op_type.name,
                'operands': to_native(op.operands),
                'const_value': to_native(op.const_value),
                'param_idx': to_native(op.param_idx),
                'coefficients': to_native(op.coefficients) if op.coefficients is not None else None,
            }
            for op in trace.operations
        ],
        'vertex_rates': to_native(trace.vertex_rates),
        'edge_probs': to_native(trace.edge_probs),
        'vertex_targets': to_native(trace.vertex_targets),
        'states': to_native(trace.states),
        'vertex_indices': to_native(trace.vertex_indices),
        'starting_vertex_idx': int(trace.starting_vertex_idx),
        'n_vertices': int(trace.n_vertices),
        'param_length': int(trace.param_length),
        'state_length': int(trace.state_length),
        'reward_length': int(trace.reward_length),
        'is_discrete': bool(trace.is_discrete),
        'metadata': to_native(trace.metadata),
    }


trace_dict = serialize_trace(trace)

# Pretty-print
print(json.dumps(trace_dict, indent=2))

In [ ]:
import gzip
import hashlib
import tempfile
from pathlib import Path

# Simulate the on-disk format: gzipped JSON + checksum
with tempfile.TemporaryDirectory() as tmpdir:
    trace_file = Path(tmpdir) / "trace.json.gz"

    # Write
    with gzip.open(trace_file, 'wt') as f:
        json.dump(trace_dict, f)

    raw_size = len(json.dumps(trace_dict).encode())
    gz_size = trace_file.stat().st_size
    checksum = hashlib.sha256(trace_file.read_bytes()).hexdigest()

    print(f"Raw JSON:     {raw_size:,} bytes")
    print(f"Gzipped:      {gz_size:,} bytes ({gz_size/raw_size:.0%})")
    print(f"SHA-256:      sha256:{checksum[:32]}...")

    # Read back
    with gzip.open(trace_file, 'rt') as f:
        loaded = json.load(f)

    print(f"\nRound-trip OK: {loaded.keys() == trace_dict.keys()}")

In [ ]:
from phasic.trace_repository import TraceRegistry

# Use the internal deserialization method
# (In normal use, get_trace() does this automatically)
registry = TraceRegistry.__new__(TraceRegistry)  # skip __init__
reconstructed = registry._deserialize_trace(loaded)

print(f"Type:            {type(reconstructed).__name__}")
print(f"Vertices:        {reconstructed.n_vertices}")
print(f"Parameters:      {reconstructed.param_length}")
print(f"Operations:      {len(reconstructed.operations)}")
print(f"Reward length:   {reconstructed.reward_length}")
print(f"Vertex indices:  {reconstructed.vertex_indices}")
print(f"State length:    {reconstructed.state_length}")

In [ ]:
from phasic.trace_elimination import instantiate_from_trace

# Instantiate with concrete parameter value
theta = np.array([2.0])  # coalescence rate parameter
concrete_graph = instantiate_from_trace(reconstructed, theta)

# Compute PDF at several time points
times = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0])
pdfs = [concrete_graph.pdf(t, granularity=100) for t in times]

print("time    pdf(t)")
print("-" * 20)
for t, p in zip(times, pdfs):
    print(f"{t:5.1f}   {p:.6f}")

## 5. Deserialization validation

The deserializer validates integrity before constructing the trace object.
This prevents corrupted or malicious data from causing silent failures.

In [ ]:
from phasic.trace_repository import TraceRegistry, _REQUIRED_TRACE_KEYS
from phasic.exceptions import PTDBackendError

registry = TraceRegistry.__new__(TraceRegistry)

# 1. Missing required keys
try:
    registry._deserialize_trace({"operations": []})
except PTDBackendError as e:
    print(f"Missing keys: {e}\n")

# 2. Invalid operation type
bad_trace = {
    "operations": [{"op_type": "BOGUS", "operands": []}],
    "vertex_rates": [0], "edge_probs": [[]], "vertex_targets": [[]],
    "states": [[0]], "starting_vertex_idx": 0, "n_vertices": 1,
    "param_length": 0, "state_length": 1,
}
try:
    registry._deserialize_trace(bad_trace)
except PTDBackendError as e:
    print(f"Invalid op_type: {e}\n")

# 3. Negative integer field
bad_trace2 = dict(bad_trace)
bad_trace2["operations"] = [{"op_type": "CONST", "operands": [], "const_value": 1.0}]
bad_trace2["n_vertices"] = -1
try:
    registry._deserialize_trace(bad_trace2)
except PTDBackendError as e:
    print(f"Negative field: {e}")

## 6. Checksum verification

Downloaded traces are verified against the `sha256:` checksum stored in the registry.
If the checksum does not match, the corrupted file is deleted and an error is raised.

In [ ]:
import tempfile, hashlib, gzip, json
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    # Create a trace file
    trace_file = Path(tmpdir) / "trace.json.gz"
    with gzip.open(trace_file, 'wt') as f:
        json.dump(trace_dict, f)

    correct_checksum = hashlib.sha256(trace_file.read_bytes()).hexdigest()
    wrong_checksum = "0" * 64

    registry = TraceRegistry.__new__(TraceRegistry)

    # Correct checksum passes silently
    registry._verify_checksum(trace_file, f"sha256:{correct_checksum}")
    print(f"Correct checksum: passed")

    # Wrong checksum raises and deletes the file
    # (Re-create the file first since _verify_checksum deletes on mismatch)
    with gzip.open(trace_file, 'wt') as f:
        json.dump(trace_dict, f)

    try:
        registry._verify_checksum(trace_file, f"sha256:{wrong_checksum}")
    except PTDBackendError as e:
        print(f"Wrong checksum:   rejected ({e})")
        print(f"File deleted:     {not trace_file.exists()}")

## 7. Download workflow

When a user calls `get_trace(trace_id)`, the following happens:

1. **Load registry** &mdash; from local cache or fetch from GitHub
2. **Look up trace_id** &mdash; get CID, checksum, and metadata
3. **Check local cache** &mdash; `~/.phasic_traces/traces/{trace_id}/trace.json.gz`
4. **Download** &mdash; `backend.get("{cid}/trace.json.gz", output_path=...)`
5. **Verify checksum** &mdash; compare `sha256:` from registry entry
6. **Decompress and deserialize** &mdash; gzip decompress, JSON parse, validate, construct `EliminationTrace`

```
~/.phasic_traces/
├── registry.json                  # cached registry from GitHub
├── traces/
│   ├── coalescent_n5_theta1/
│   │   └── trace.json.gz          # cached by trace_id
│   └── coalescent_n10_theta1/
│       └── trace.json.gz
└── by_hash/
    └── a1b2c3d4.../               # cached by graph structure hash
        └── trace.json.gz
```

The `by_hash/` directory enables automatic trace discovery: compute a graph's
structure hash and check if a pre-computed trace already exists.

## 8. Publish workflow

Publishing creates a directory package on IPFS and prints instructions
for submitting a PR to the registry.

### Package structure on IPFS

```
{trace_id}/
├── trace.json.gz          # gzipped serialized EliminationTrace
├── metadata.json           # human-readable metadata
├── checksum.sha256         # SHA-256 of trace.json.gz
├── construction_code.py    # (optional) code to rebuild the graph
└── example.py              # (optional) usage example
```

### Workflow

```python
from phasic import TraceRegistry
from phasic.trace_elimination import record_elimination_trace

# 1. Build graph and record trace
trace = record_elimination_trace(graph, param_length=1)

# 2. Publish
registry = TraceRegistry()
cid = registry.publish_trace(
    trace=serialize_trace(trace),
    trace_id="my_coalescent_model",
    metadata={
        "model_type": "coalescent",
        "domain": "population-genetics",
        "param_length": 1,
        "description": "Kingman coalescent for n=5",
        "author": "Name <email>",
        "license": "MIT"
    },
    construction_code=open("build_model.py").read(),  # optional
    submit_pr=True  # prints PR instructions
)
# Returns: CID of published package
```

### Adding to the public registry

After publishing, the user forks the registry repo on GitHub, adds the entry
printed by `submit_pr=True` to `registry.json`, and submits a pull request.
The maintainer pins the CID to IPFS pinning services and merges.

## 9. HTTP retry protocol

All HTTP requests (registry updates, gateway downloads) use exponential-backoff
retry with the `_request_with_retry()` helper:

- **3 attempts** by default
- **Retries on**: `ConnectionError`, HTTP 5xx status codes
- **Does not retry**: HTTP 4xx (client errors), non-connection exceptions
- **Backoff**: 1s, 2s, 4s (doubles each attempt)

In [ ]:
from unittest import mock
import requests
from phasic.trace_repository import _request_with_retry

# Simulate: first attempt fails, second succeeds
ok_response = mock.Mock()
ok_response.status_code = 200
ok_response.raise_for_status = mock.Mock()
ok_response.content = b"trace data"

with mock.patch('requests.get') as mock_get:
    mock_get.side_effect = [
        requests.ConnectionError("network blip"),
        ok_response,
    ]
    with mock.patch('time.sleep'):  # skip actual delay
        result = _request_with_retry("https://example.com/trace", max_retries=3, backoff_base=0.01)

print(f"Attempts: {mock_get.call_count}")
print(f"Result:   {result.content}")

## Summary

| Component | Format / Protocol | Purpose |
|-----------|-------------------|--------|
| `TransportBackend` | Python ABC | Pluggable storage (IPFS, S3, GCS, local) |
| `registry.json` | JSON on GitHub | Index: trace IDs, CIDs, checksums, metadata |
| `trace.json.gz` | Gzipped JSON | Serialized `EliminationTrace` |
| CID | IPFS content hash | Content-addressed identifier (CIDv0/v1) |
| Checksum | `sha256:<hex>` | Integrity verification |
| Operations | `OpType` enum | Elimination trace instructions (CONST, PARAM, DOT, ADD, ...) |
| Cache | `~/.phasic_traces/` | Local cache by trace_id and by graph hash |
| HTTP retry | Exponential backoff | 3 retries, 1s/2s/4s backoff, retries on 5xx/ConnectionError |

### Key design decisions

- **Content-addressed storage**: CIDs guarantee that downloaded content matches what was published.
  The checksum adds a second layer of verification.
- **Lazy backend initialization**: The `TransportBackend` is not created until first use,
  so `TraceRegistry(auto_update=False)` never contacts IPFS.
- **Progressive enhancement**: Works with zero configuration (HTTP gateways),
  gets faster with IPFS daemon.
- **Pluggable backends**: The `TransportBackend` ABC allows future migration to
  cloud storage without changing the registry or client APIs.
- **Offline-first**: Local cache is checked before any network call.